# Quantization Lab: one model, 16-bit vs every quant (Kaggle master notebook)

**Repo:** https://github.com/jprakash-1/model-quantization-exp · **Model:** `Qwen/Qwen3-4B-Instruct-2507` (4.02B params, non-thinking, greedy). FP16 = 8 GB, so it fits the P100's 16 GB with room for batched generation. Each quantized variant is compared to the 16-bit original on:

| | Metric | How |
|---|---|---|
| Memory | weights size, **peak GPU memory** | `get_memory_footprint()` / file size, NVML sampling |
| Speed | prefill tok/s, **decode tok/s**, TTFT for a 512-token prompt | single stream, median of 3 |
| Drift | **KL divergence** vs baseline, top-1 token agreement, perplexity | wikitext-2, 20 × 512 tokens |
| Quality | **GSM8K** strict/flexible, **ARC-Challenge**, answers flipped vs baseline | 250 questions each, greedy |

**Two tracks**, each compared only against its own baseline:
1. **Hugging Face Transformers**: bitsandbytes INT8 / NF4 / FP4, Unsloth dynamic 4-bit, HQQ 4/3/2-bit, torchao, FP8
2. **llama.cpp GGUF**: Q8_0 down to Q2_K, with and without Unsloth's imatrix, plus Unsloth Dynamic `UD-*` quants

### Kaggle settings (right sidebar)
- **Accelerator:** **GPU P100** (16 GB HBM2, Pascal, compute capability 6.0). This notebook is set up for it; it also works on a T4.
- **What the P100 means for this lab:**
  - The baseline is **FP16** (Pascal has no BF16), and **FP8 will fail**, which is a result in itself.
  - Recent PyTorch builds dropped Pascal kernels. Cell 1 detects this and installs the CUDA 12.6 build instead.
  - llama.cpp's fastest int8 kernels need `dp4a` (compute capability 6.1+), so GGUF quants run on slower fallback paths. Compare GGUF speeds with each other, not with numbers you see online.
- **Internet:** On
- For an unattended run: **Save Version → Save & Run All (Commit)**. It runs in the background for up to 12 h, and `results/` lands in the notebook output.

Every variant writes `results/<track>-<variant>/metrics.json`. **Re-running skips finished variants**, so a timeout costs nothing. A variant that fails writes `error.txt` and the loop continues.

In [ ]:
# 1. Get the latest code + install
REPO_URL = "https://github.com/jprakash-1/model-quantization-exp.git"
REPO_DIR = "/kaggle/working/model-quantization-exp"

import os, subprocess, sys

# Stop early if the GPU driver is too old: PyTorch's CUDA 12 builds need driver >= 525.
driver = subprocess.run(["nvidia-smi", "--query-gpu=name,driver_version", "--format=csv,noheader"],
                        capture_output=True, text=True).stdout.strip()
print("GPU, driver:", driver)
if int(driver.split(",")[-1].strip().split(".")[0]) < 525:
    raise SystemExit("Driver < 525 (CUDA 10/11 era): no modern PyTorch or transformers can run here. "
                     "Restart the session with a different accelerator, or check that GPU is enabled.")

if os.path.exists(REPO_DIR):
    !git -C {REPO_DIR} pull -q --ff-only
else:
    !git clone -q {REPO_URL} {REPO_DIR}
os.chdir(REPO_DIR)
!git log --oneline -1
!pip install -q -U -r requirements-kaggle.txt

# P100 = compute capability 6.0. PyTorch's CUDA 12.8+ builds ship no kernels for it, CUDA 12.6 builds do.
# Checked in a subprocess so torch is not imported here yet, and swapping it needs no restart.
CHECK = "import torch; m, n = torch.cuda.get_device_capability(0); print(f'sm_{m}{n}' in torch.cuda.get_arch_list(), torch.__version__, f'sm_{m}{n}')"
ok, ver, arch = subprocess.run([sys.executable, "-c", CHECK], capture_output=True, text=True).stdout.split()
print(f"torch {ver} has {arch} kernels: {ok}")
if ok != "True":
    !pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu126
    print("reinstalled:", subprocess.run([sys.executable, "-c", CHECK], capture_output=True, text=True).stdout)

In [ ]:
# 2. Configure (env var first: it must be set before torch is imported)
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

from qlab import config
config.N_QUESTIONS = 250     # 100 for a quick first pass (~3x faster)
# config.set_model("Qwen/Qwen3-1.7B")   # optional: ~2.5x faster run on a smaller model

print("model:", config.MODEL_ID)
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv
!df -h /tmp /kaggle/working | tail -2

In [ ]:
# 3. Data: wikitext-2 (drift), GSM8K + ARC-Challenge (tasks)
from qlab import data
data.download(n=config.N_QUESTIONS)

## Track 1: Hugging Face Transformers

The **baseline runs first**: it saves the reference log-probs that every other variant's KL divergence is measured against.

| Variant | What it is | Lesson |
|---|---|---|
| `fp16` / `bf16` | original weights (BF16 needs compute capability ≥ 8; a P100 or T4 gets FP16) | reference |
| `bnb-int8` | LLM.int8(): INT8 weights, outlier features kept in 16-bit | "8-bit is free", but is it fast? |
| `bnb-nf4` | QLoRA's 4-bit NormalFloat, block size 64 | standard 4-bit |
| `bnb-nf4-dq` | NF4 plus quantized quantization scales | how much do the scales cost? |
| `bnb-fp4` | 4-bit float instead of NF4 | does the 4-bit *code book* matter? |
| `unsloth-dyn4` | Unsloth's NF4 with sensitive layers left in 16-bit | mixed precision vs uniform NF4 |
| `hqq4/3/2` | Half-Quadratic Quantization, no calibration data, group 64 | where is the cliff? |
| `torchao-int8` | int8 weight-only *(optional)* | |
| `fp8` | Qwen's official FP8 checkpoint *(needs compute capability ≥ 8.9, so it fails on a P100 (6.0) and a T4 (7.5))* | your GPU decides the format |

In [ ]:
from qlab import run
from qlab.hf_backend import BASELINE

HF_VARIANTS = [BASELINE, "bnb-int8", "bnb-nf4", "bnb-nf4-dq", "bnb-fp4", "unsloth-dyn4", "hqq4", "hqq3", "hqq2", "fp8"]
run.run_all("hf", HF_VARIANTS)

In [ ]:
# Optional extra: needs one more library, which must match the installed torch version.
# !pip install -q torchao && python -c "import torchao"
# run.run("hf", "torchao-int8")

## Track 2: llama.cpp / GGUF

llama.cpp is built from source with CUDA for this GPU (~10–15 min, once per session). The quants are made here from Unsloth's 16-bit GGUF. `*-imat` uses Unsloth's importance matrix, and `UD-*` files are downloaded from Unsloth. The baseline is F16, because the P100 has no BF16. On a P100 (no `dp4a`), quantized matmuls take slower fallback paths, so watch whether smaller quants are actually faster here.

| Variant | Lesson |
|---|---|
| `F16` | reference (Unsloth publishes F16 for this model, so it is also the quant source) |
| `Q8_0`, `Q6_K`, `Q5_K_M` | the near-lossless zone |
| `Q4_K_M` vs `Q4_K_M-imat` vs `UD-Q4_K_XL` | what calibration, then per-layer mixing, buy at 4-bit |
| `IQ4_XS-imat` | non-uniform 4-bit code book |
| `Q3_K_M-imat` | the 3-bit cliff |
| `Q2_K` vs `Q2_K-imat` vs `UD-Q2_K_XL` | can calibration and mixing rescue 2-bit? |

In [ ]:
from qlab import gguf_backend as gg
gg.build()      # llama.cpp + CUDA
gg.prepare()    # BF16 GGUF + imatrix download, F16 baseline

In [ ]:
GGUF_VARIANTS = ["F16", "Q8_0", "Q6_K", "Q5_K_M", "Q4_K_M", "Q4_K_M-imat", "UD-Q4_K_XL",
                 "IQ4_XS-imat", "Q3_K_M-imat", "Q2_K", "Q2_K-imat", "UD-Q2_K_XL"]
run.run_all("gguf", GGUF_VARIANTS)

## Report

In [ ]:
from qlab import report
import pandas as pd
pd.set_option("display.width", 250, "display.max_columns", 30)

df, errors = report.table()
display(df[["track", "variant", "bpw", "size_gb", "peak_gpu_gb", "ttft_ms", "decode_tps",
            "ppl", "mean_kld", "top1_agree", "gsm8k_strict", "gsm8k_flexible", "arc", "lost_vs_base", "gained_vs_base"]])
for name, err in errors.items():
    print(f"FAILED {name}: {err}")
report.plots(df);

In [ ]:
# Look at the answers a variant got wrong that the baseline got right
import json
def lost(track, variant, k=3):
    base = {"hf": BASELINE, "gguf": "F16"}[track]
    b = json.load(open(config.RESULTS / f"{track}-{base}" / "eval.json"))["gsm8k"]
    q = json.load(open(config.RESULTS / f"{track}-{variant}" / "eval.json"))["gsm8k"]
    for x, y in [(x, y) for x, y in zip(b, q) if x["flexible"] and not y["flexible"]][:k]:
        print(f"--- question {x['i']}\nBASELINE: ...{x['reply'][-200:]}\n{variant.upper()}: ...{y['reply'][-200:]}\n")

lost("hf", "hqq3")

## Questions to answer from the table

1. Is 8-bit (`bnb-int8`, `Q8_0`) indistinguishable from the baseline? Look at mean KLD, net flips, and whether `bnb-int8` is actually *slower*.
2. At 4-bit, what matters more: the code book (NF4 vs FP4), the scales (`-dq`), calibration (`-imat`), or per-layer mixing (`unsloth-dyn4`, `UD-*`)?
3. Does perplexity move under 1% while KLD and GSM8K clearly move? (Ledger: "perplexity hides most of the damage")
4. Does GSM8K drop before ARC does? (Ledger: "the benchmark decides what you see")
5. Is *strict minus flexible* bigger for some quants, meaning they break the answer format before the math?
6. Where is the cliff for a 4B model: 3-bit or 2-bit? Does imatrix or UD move it?
7. Does decode speed follow weight size? Which formats are smaller but *not* faster (dequantization overhead)?
8. Bits per weight vs peak GPU memory: which variants use more memory than their bits suggest, and why?
9. Your pick: which single variant would you deploy on a 16 GB GPU, and why?

In [ ]:
# Wrap up: free the big files (weights, llama.cpp build, KLD refs). results/ stays in the notebook output.
run.cleanup()
!du -sh results && ls results

In [ ]:
# Optional: push results back to your repo. Add a GitHub token as a Kaggle Secret named GITHUB_TOKEN first.
# from kaggle_secrets import UserSecretsClient
# token = UserSecretsClient().get_secret("GITHUB_TOKEN")
# !git config user.email "you@example.com" && git config user.name "kaggle-run"
# !git -C {REPO_DIR} add results && git -C {REPO_DIR} commit -qm "Kaggle run results" && git -C {REPO_DIR} push -q https://{token}@{REPO_URL.removeprefix("https://")} HEAD